# Preparação da base — merge e tratamento (Olist)

Este notebook junta as tabelas do dataset **Brazilian E-Commerce (Olist)** em uma única base,
tratando valores nulos e duplicados, e gera os dois arquivos que alimentam o pipeline RAG:

| Arquivo gerado | Conteúdo |
|---|---|
| `data/processed/base_avaliacoes_completa.csv` | todas as avaliações (com e sem comentário), já com os metadados dos demais arquivos |
| `data/processed/base_rag.csv` | só avaliações com comentário de texto, prontas para virar documentos do RAG |

**Tabelas usadas e chaves de junção** (conforme o diagrama do dataset):

```
reviews ──order_id──▶ orders ──customer_id──▶ customers ──zip_code_prefix──▶ geolocation
                         │
                         └──order_id──▶ order_items ──product_id──▶ products     (opcional, ver Seção 5)
```

> `products` não se liga direto a `orders`: a ponte é a tabela `olist_order_items_dataset.csv`.
> Se ela estiver na pasta, a Seção 5 enriquece a base com a categoria do produto; se não estiver, o notebook segue sem ela.

## Seção 0 — Setup

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

PASTA_RAW = Path("../data/raw")              # CSVs baixados do Kaggle
PASTA_PROCESSED = Path("../data/processed")  # saídas deste notebook
PASTA_PROCESSED.mkdir(parents=True, exist_ok=True)

MIN_CHARS_COMENTARIO = 5      # comentários com menos caracteres que isso são tratados como "sem comentário"
DEDUP_TEXTOS_RAG = True       # remove textos idênticos na base do RAG (guardando a contagem em n_ocorrencias)

log = {}  # registro de tudo o que foi removido/tratado (útil para justificar no README)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
print("Setup pronto.")

Setup pronto.


## Seção 1 — Leitura e diagnóstico inicial

Os CEPs (`zip_code_prefix`) são lidos como **texto**: eles são chaves de junção, não números.
Como os CSVs do Kaggle guardam esses prefixos sem o zero à esquerda, padronizamos com 5 dígitos nas duas pontas.

In [5]:
reviews     = pd.read_csv(PASTA_RAW / "olist_order_reviews_dataset.csv")
orders      = pd.read_csv(PASTA_RAW / "olist_orders_dataset.csv")
customers   = pd.read_csv(PASTA_RAW / "olist_customers_dataset.csv", dtype={"customer_zip_code_prefix": str})
geolocation = pd.read_csv(PASTA_RAW / "olist_geolocation_dataset.csv", dtype={"geolocation_zip_code_prefix": str})
products    = pd.read_csv(PASTA_RAW / "olist_products_dataset.csv")

tabelas = {
    "reviews": reviews, "orders": orders, "customers": customers,
    "geolocation": geolocation, "products": products,
}


def diagnostico(nome, df):
    print("=" * 72)
    print(f"{nome}: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
    print(f"  linhas 100% duplicadas: {df.duplicated().sum():,}")
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0]
    if len(nulos):
        print("  nulos por coluna:")
        for col, n in nulos.items():
            print(f"    {col:35s} {n:>9,}  ({n / len(df):.1%})")
    else:
        print("  sem valores nulos")


for nome, df in tabelas.items():
    diagnostico(nome, df)

reviews: 99,224 linhas x 7 colunas
  linhas 100% duplicadas: 0
  nulos por coluna:
    review_comment_title                   87,656  (88.3%)
    review_comment_message                 58,247  (58.7%)
orders: 99,441 linhas x 8 colunas
  linhas 100% duplicadas: 0
  nulos por coluna:
    order_approved_at                         160  (0.2%)
    order_delivered_carrier_date            1,783  (1.8%)
    order_delivered_customer_date           2,965  (3.0%)
customers: 99,441 linhas x 5 colunas
  linhas 100% duplicadas: 0
  sem valores nulos
geolocation: 1,000,163 linhas x 5 colunas
  linhas 100% duplicadas: 261,831
  sem valores nulos
products: 32,951 linhas x 9 colunas
  linhas 100% duplicadas: 0
  nulos por coluna:
    product_category_name                     610  (1.9%)
    product_name_lenght                       610  (1.9%)
    product_description_lenght                610  (1.9%)
    product_photos_qty                        610  (1.9%)
    product_weight_g                          

## Seção 2 — Limpeza tabela por tabela

### 2.1 `reviews` (a tabela central)

Decisões:
- **Duplicatas exatas** são removidas.
- **Um pedido pode ter mais de uma avaliação** (e o `review_id` também se repete entre pedidos).
  Mantemos **a avaliação mais recente de cada `order_id`**, o que deixa `order_id` como chave única.
- **Nulos em título e mensagem** são esperados (a maioria dos clientes só dá a nota). Não são removidos aqui:
  viram texto vazio e a base completa mantém essas linhas; só a base do RAG filtra por comentário.
- Textos têm quebras de linha e espaços repetidos normalizados.

In [6]:
n0 = len(reviews)
reviews = reviews.drop_duplicates()
log["reviews: linhas 100% duplicadas removidas"] = n0 - len(reviews)

# datas
for col in ["review_creation_date", "review_answer_timestamp"]:
    reviews[col] = pd.to_datetime(reviews[col], errors="coerce")

# nota válida (1 a 5)
n0 = len(reviews)
reviews = reviews[reviews["review_score"].between(1, 5)]
log["reviews: notas fora de 1-5 removidas"] = n0 - len(reviews)


def limpar_texto(serie):
    """Nulo vira string vazia; quebras de linha/tabs e espaços repetidos viram um espaço só."""
    return (serie.fillna("").astype(str)
            .str.replace(r"[\r\n\t]+", " ", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip())


log["reviews: títulos nulos (viraram vazio)"] = int(reviews["review_comment_title"].isna().sum())
log["reviews: mensagens nulas (viraram vazio)"] = int(reviews["review_comment_message"].isna().sum())
reviews["review_comment_title"] = limpar_texto(reviews["review_comment_title"])
reviews["review_comment_message"] = limpar_texto(reviews["review_comment_message"])

# um pedido com várias avaliações -> fica a mais recente
pedidos_com_varias = reviews["order_id"].duplicated(keep=False).sum()
reviews = reviews.sort_values(["order_id", "review_answer_timestamp", "review_creation_date"],
                              na_position="first")
n0 = len(reviews)
reviews = reviews.drop_duplicates(subset="order_id", keep="last")
log["reviews: avaliações antigas de pedidos com mais de uma avaliação"] = n0 - len(reviews)

assert reviews["order_id"].is_unique
print(f"reviews limpa: {len(reviews):,} linhas (uma por pedido)")

reviews limpa: 98,673 linhas (uma por pedido)


### 2.2 `orders`

Os nulos em `order_approved_at`, `order_delivered_carrier_date` e `order_delivered_customer_date` **não são erro**:
representam pedidos cancelados, indisponíveis ou ainda não entregues. Por isso **não são imputados**
(inventar uma data de entrega distorceria os indicadores). Em vez disso, criamos variáveis derivadas de entrega,
que são úteis como metadados para perguntas sobre atraso.

In [7]:
n0 = len(orders)
orders = orders.drop_duplicates(subset="order_id")
log["orders: pedidos duplicados removidos"] = n0 - len(orders)

colunas_data = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
for col in colunas_data:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

dia = pd.Timedelta(days=1)
orders["tempo_entrega_dias"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]) / dia
orders["dias_atraso"] = (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]) / dia
# True/False só quando existe data de entrega; caso contrário fica <NA> (não sabemos)
orders["entregue_com_atraso"] = (orders["dias_atraso"] > 0).where(orders["dias_atraso"].notna()).astype("boolean")

print(f"orders limpa: {len(orders):,} linhas")
print("\nStatus dos pedidos:")
print(orders["order_status"].value_counts().to_string())

orders limpa: 99,441 linhas

Status dos pedidos:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2


### 2.3 `customers`

`customer_id` é único por pedido; `customer_unique_id` se repete (mesmo cliente comprando mais de uma vez) —
isso é legítimo e **não é duplicata**.

In [8]:
n0 = len(customers)
customers = customers.drop_duplicates(subset="customer_id")
log["customers: customer_id duplicados removidos"] = n0 - len(customers)

customers["customer_zip_code_prefix"] = customers["customer_zip_code_prefix"].str.zfill(5)
customers["customer_city"] = customers["customer_city"].str.strip().str.lower()
customers["customer_state"] = customers["customer_state"].str.strip().str.upper()

print(f"customers limpa: {len(customers):,} linhas")

customers limpa: 99,441 linhas


### 2.4 `geolocation`

Esta tabela tem **muitas linhas por CEP** (várias coordenadas para o mesmo prefixo) e algumas coordenadas
fora do Brasil. Para que a junção **não multiplique linhas**, fazemos:
1. remover duplicatas exatas;
2. descartar coordenadas fora do retângulo geográfico do Brasil;
3. **agregar por CEP**: mediana de latitude/longitude (robusta a outliers) e cidade/estado mais frequentes.

O resultado tem exatamente uma linha por `zip_code_prefix`.

In [9]:
n0 = len(geolocation)
geolocation = geolocation.drop_duplicates()
log["geolocation: linhas 100% duplicadas removidas"] = n0 - len(geolocation)

geolocation["geolocation_zip_code_prefix"] = geolocation["geolocation_zip_code_prefix"].str.zfill(5)

dentro_do_brasil = (geolocation["geolocation_lat"].between(-34, 6)
                    & geolocation["geolocation_lng"].between(-74, -34))
log["geolocation: coordenadas fora do Brasil removidas"] = int((~dentro_do_brasil).sum())
geolocation = geolocation[dentro_do_brasil]


def moda(serie):
    m = serie.mode()
    return m.iat[0] if len(m) else np.nan


geo_cep = (geolocation.groupby("geolocation_zip_code_prefix")
           .agg(geo_lat=("geolocation_lat", "median"),
                geo_lng=("geolocation_lng", "median"),
                geo_city=("geolocation_city", moda),
                geo_state=("geolocation_state", moda))
           .reset_index())

assert geo_cep["geolocation_zip_code_prefix"].is_unique
print(f"geolocation agregada: {len(geo_cep):,} CEPs únicos")

C:\Users\monef\AppData\Local\Temp\ipykernel_16560\251007736.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  geolocation["geolocation_zip_code_prefix"] = geolocation["geolocation_zip_code_prefix"].str.zfill(5)


geolocation agregada: 19,010 CEPs únicos


### 2.5 `products`

Só duas colunas interessam para o RAG (`product_id` e a categoria). Categoria nula vira `"sem_categoria"`
(rótulo explícito é melhor que deixar nulo, porque permite filtrar e contar).
As medidas físicas (peso, dimensões) não entram na base final.

In [10]:
n0 = len(products)
products = products.drop_duplicates(subset="product_id")
log["products: product_id duplicados removidos"] = n0 - len(products)

log["products: categoria nula (virou 'sem_categoria')"] = int(products["product_category_name"].isna().sum())
products["product_category_name"] = products["product_category_name"].fillna("sem_categoria")

print(f"products limpa: {len(products):,} linhas")

products limpa: 32,951 linhas


## Seção 3 — Merge

Todas as junções são `left` a partir de `reviews` (cada avaliação deve continuar existindo) e usam
`validate=` para **garantir que nenhuma linha foi multiplicada** por chave repetida.

In [11]:
n_reviews = len(reviews)

base = reviews.merge(orders, on="order_id", how="left", validate="one_to_one")
base = base.merge(customers, on="customer_id", how="left", validate="many_to_one")
base = base.merge(geo_cep, left_on="customer_zip_code_prefix",
                  right_on="geolocation_zip_code_prefix", how="left", validate="many_to_one")
base = base.drop(columns=["geolocation_zip_code_prefix"])

assert len(base) == n_reviews, "o merge alterou o número de linhas!"
assert base["order_id"].is_unique

print(f"base após merge: {len(base):,} linhas x {base.shape[1]} colunas")
print(f"avaliações sem pedido correspondente em orders: {base['order_status'].isna().sum():,}")
print(f"avaliações sem coordenadas (CEP não encontrado em geolocation): {base['geo_lat'].isna().sum():,}")

base após merge: 98,673 linhas x 25 colunas
avaliações sem pedido correspondente em orders: 0
avaliações sem coordenadas (CEP não encontrado em geolocation): 277


## Seção 4 — Categoria do produto (opcional, via `order_items`)

A categoria é o metadado mais útil para o desafio adicional ("o que os clientes elogiam em cada categoria?").
Como um pedido pode ter **vários itens de categorias diferentes**, agregamos por pedido:
`categoria_principal` (a mais frequente), `categorias` (todas) e `n_itens`.

In [13]:
itens = pd.read_csv(PASTA_RAW / "olist_order_items_dataset.csv")

n0 = len(itens)
itens = itens.drop_duplicates()
log["order_items: linhas 100% duplicadas removidas"] = n0 - len(itens)

# traz a categoria de cada item (cada product_id aparece uma vez em products)
itens = itens.merge(products[["product_id", "product_category_name"]],
                    on="product_id", how="left", validate="many_to_one")

# produto sem categoria (nulo em products) ou sem cadastro em products
log["order_items: itens sem categoria (viraram 'sem_categoria')"] = int(itens["product_category_name"].isna().sum())
itens["product_category_name"] = itens["product_category_name"].fillna("sem_categoria")

# um pedido pode ter vários itens: agrega para uma linha por pedido
por_pedido = (itens.groupby("order_id")
              .agg(n_itens=("order_item_id", "count"),
                   valor_itens=("price", "sum"),
                   valor_frete=("freight_value", "sum"),
                   categoria_principal=("product_category_name", moda),
                   categorias=("product_category_name", lambda s: " | ".join(sorted(set(s)))))
              .reset_index())

base = base.merge(por_pedido, on="order_id", how="left", validate="one_to_one")

# pedidos que não têm nenhum item registrado em order_items
for col in ["categoria_principal", "categorias"]:
    base[col] = base[col].fillna("sem_itens")

assert len(base) == n_reviews, "o merge com order_items alterou o número de linhas!"

print(f"categoria adicionada. Pedidos sem itens registrados: {(base['categoria_principal'] == 'sem_itens').sum():,}")
print("\nTop 10 categorias principais:")
print(base["categoria_principal"].value_counts().head(10).to_string())

categoria adicionada. Pedidos sem itens registrados: 756

Top 10 categorias principais:
categoria_principal
cama_mesa_banho           9268
beleza_saude              8760
esporte_lazer             7632
informatica_acessorios    6619
moveis_decoracao          6273
utilidades_domesticas     5739
relogios_presentes        5540
telefonia                 4144
automotivo                3871
brinquedos                3833


## Seção 5 — Texto da avaliação e tratamento final

- `texto_avaliacao` = título + mensagem (quando os dois existem), que é o campo que será embutido no RAG.
- `tem_comentario` = o texto tem ao menos `MIN_CHARS_COMENTARIO` caracteres.
- Colunas de texto restantes com nulo recebem um rótulo explícito; **coordenadas ausentes ficam nulas**
  (não faz sentido inventar latitude).

In [14]:
def juntar(titulo, mensagem):
    if titulo and mensagem:
        return f"{titulo}. {mensagem}"
    return titulo or mensagem


base["texto_avaliacao"] = [juntar(t, m) for t, m in
                           zip(base["review_comment_title"], base["review_comment_message"])]
base["tem_comentario"] = base["texto_avaliacao"].str.len() >= MIN_CHARS_COMENTARIO

# rótulos explícitos para categóricos nulos (ex.: avaliação sem pedido correspondente)
for col in ["order_status", "customer_city", "customer_state"]:
    base[col] = base[col].fillna("desconhecido")

log["base: avaliações com comentário utilizável"] = int(base["tem_comentario"].sum())
log["base: avaliações só com nota (sem comentário)"] = int((~base["tem_comentario"]).sum())

print(f"com comentário: {base['tem_comentario'].sum():,} ({base['tem_comentario'].mean():.1%})")
print("\nDistribuição das notas (todas as avaliações):")
print(base["review_score"].value_counts().sort_index().to_string())
print("\nNulos restantes na base completa:")
nulos = base.isna().sum()
print(nulos[nulos > 0].to_string())

com comentário: 41,497 (42.1%)

Distribuição das notas (todas as avaliações):
review_score
1    11363
2     3131
3     8133
4    19038
5    57008

Nulos restantes na base completa:
order_approved_at                 155
order_delivered_carrier_date     1746
order_delivered_customer_date    2843
tempo_entrega_dias               2843
dias_atraso                      2843
entregue_com_atraso              2843
geo_lat                           277
geo_lng                           277
geo_city                          277
geo_state                         277
n_itens                           756
valor_itens                       756
valor_frete                       756


## Seção 6 — Base do RAG

Só entram avaliações **com comentário**. Como muitos comentários curtos se repetem literalmente
("Ótimo", "Recomendo"), e isso encheria o top-k de recuperação com o mesmo texto, removemos textos idênticos
(comparando em minúsculas e sem pontuação) e **guardamos a frequência em `n_ocorrencias`**, para não perder
a informação de recorrência. Para desligar isso, use `DEDUP_TEXTOS_RAG = False` no setup.

In [15]:
rag = base[base["tem_comentario"]].copy()

rag["texto_norm"] = (rag["texto_avaliacao"].str.lower()
                     .str.replace(r"[^\w\s]", "", regex=True)
                     .str.replace(r"\s+", " ", regex=True)
                     .str.strip())
rag["n_ocorrencias"] = rag.groupby("texto_norm")["order_id"].transform("size")

if DEDUP_TEXTOS_RAG:
    n0 = len(rag)
    rag = rag.sort_values("review_creation_date").drop_duplicates(subset="texto_norm", keep="last")
    log["rag: textos idênticos removidos (frequência guardada em n_ocorrencias)"] = n0 - len(rag)

rag["doc_id"] = rag["order_id"]   # order_id é único na base; review_id NÃO é

colunas_rag = [
    "doc_id", "review_id", "order_id", "texto_avaliacao", "review_score", "review_creation_date",
    "n_ocorrencias", "order_status", "order_purchase_timestamp", "tempo_entrega_dias", "dias_atraso",
    "entregue_com_atraso", "customer_state", "customer_city", "geo_lat", "geo_lng",
    "categoria_principal", "categorias", "n_itens",
]
colunas_rag = [c for c in colunas_rag if c in rag.columns] 
rag = rag[colunas_rag].rename(columns={"review_score": "nota", "texto_avaliacao": "texto",
                                       "review_creation_date": "data_avaliacao",
                                       "order_purchase_timestamp": "data_compra"})
rag = rag.reset_index(drop=True)

assert rag["doc_id"].is_unique
print(f"base do RAG: {len(rag):,} documentos")
print("\nTamanho dos textos (caracteres):")
print(rag["texto"].str.len().describe().round(0).to_string())
rag.head(3)

base do RAG: 35,027 documentos

Tamanho dos textos (caracteres):
count    35027.0
mean        81.0
std         53.0
min          5.0
25%         40.0
50%         65.0
75%        108.0
max        229.0


,doc_id,review_id,order_id,texto,nota,data_avaliacao,n_ocorrencias,order_status,data_compra,tempo_entrega_dias,dias_atraso,entregue_com_atraso,customer_state,customer_city,geo_lat,geo_lng,categoria_principal,categorias,n_itens
0,809a282bbd5dbcabb6f2f724fca862ec,69ac6a27fde9855ebeaaecac0f78058b,809a282bbd5dbcabb6f2f724fca862ec,MEU PEDIDO NÃO FOI ENTREGUE E NÃO FOI DADA NEN...,1,2016-10-02,1,canceled,2016-09-13 15:24:19,NaN,NaN,<NA>,SP,sao jose dos campos,-23.200115,-45.944485,sem_itens,sem_itens,NaN
1,bfbd0f9bdef84302105ad712db648a6c,6916ca4502d6d3bfd39818759d55d536,bfbd0f9bdef84302105ad712db648a6c,nao recebi o produto e nem resposta da empresa,1,2016-10-06,1,delivered,2016-09-15 12:16:38,54.813194,36.324745,True,SP,sao joaquim da barra,-20.585521,-47.863166,beleza_saude,beleza_saude,3.0
2,e5215415bb6f76fe3b7cb68103a0d1c0,49f695dffa457eaba90d388a5c37e942,e5215415bb6f76fe3b7cb68103a0d1c0,"PRODUTO NÃO CHEGOU,E JÁ PASSOU O PRAZO DE ENTREGA",1,2016-10-09,1,canceled,2016-10-22 08:25:27,NaN,NaN,<NA>,CE,paracuru,-3.416460,-39.030822,sem_itens,sem_itens,NaN


## Seção 7 — Salvando e resumo do tratamento

In [16]:
base.to_csv(PASTA_PROCESSED / "base_avaliacoes_completa.csv", index=False, encoding="utf-8")
rag.to_csv(PASTA_PROCESSED / "base_rag.csv", index=False, encoding="utf-8")
print("Salvo em data/processed/: base_avaliacoes_completa.csv e base_rag.csv\n")

print("RESUMO DO TRATAMENTO")
print("-" * 72)
for descricao, valor in log.items():
    print(f"  {descricao:70s} {valor:>9,}")

Salvo em data/processed/: base_avaliacoes_completa.csv e base_rag.csv

RESUMO DO TRATAMENTO
------------------------------------------------------------------------
  reviews: linhas 100% duplicadas removidas                                      0
  reviews: notas fora de 1-5 removidas                                           0
  reviews: títulos nulos (viraram vazio)                                    87,656
  reviews: mensagens nulas (viraram vazio)                                  58,247
  reviews: avaliações antigas de pedidos com mais de uma avaliação             551
  orders: pedidos duplicados removidos                                           0
  customers: customer_id duplicados removidos                                    0
  geolocation: linhas 100% duplicadas removidas                            261,831
  geolocation: coordenadas fora do Brasil removidas                             33
  products: product_id duplicados removidos                                      0
  pro